In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error


In [2]:
df = pd.read_csv("housing.csv")

df['income_cat'] = pd.cut(df['median_income'], 
                          bins = [0,1.5,3.0,4.5,6.0,np.inf],
                         labels = [1, 2, 3, 4, 5])

split = StratifiedShuffleSplit( n_splits= 1, test_size= 0.2, random_state= 42)

for train_index, test_index in split.split(df, df['income_cat']):
    strata_train_set = df.loc[train_index].drop(['income_cat'],axis=1)
    strata_test_set = df.loc[test_index].drop(['income_cat'],axis=1)

housing = strata_train_set.copy()

housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

num_attribs = housing.drop(['ocean_proximity'], axis =1).columns.tolist()
cat_attribs = ['ocean_proximity']

In [3]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,INLAND
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,NEAR OCEAN
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,INLAND
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,NEAR OCEAN
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900,<1H OCEAN
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139,INLAND
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797,<1H OCEAN
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964,<1H OCEAN


In [5]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown = "ignore"))
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs)
])

housing_prep = full_pipeline.fit_transform(housing)
print(housing.shape)

(16512, 9)


In [6]:
lin_reg = LinearRegression()
lin_reg.fit(housing_prep, housing_labels)
lin_preds = lin_reg.predict(housing_prep)
lin_rmse = root_mean_squared_error(housing_labels, lin_preds, )
print("Linear Regression RMSE:", lin_rmse)

Linear Regression RMSE: 69050.56219504568


In [16]:
#  DecisionTreeRegressor
dec_reg =  DecisionTreeRegressor()
dec_reg.fit(housing_prep, housing_labels)
dec_preds = dec_reg.predict(housing_prep)
dec_rmse = root_mean_squared_error(housing_labels, dec_preds)
print("DecisionTreeRegressor RMSE:", dec_rmse) # output is 0.0 because of over fitting

DecisionTreeRegressor RMSE: 0.0


In [17]:
# RandomForestRegressor
ran_reg = RandomForestRegressor()
ran_reg.fit(housing_prep, housing_labels)
ran_preds = ran_reg.predict(housing_prep)
ran_rmse = root_mean_squared_error(housing_labels, ran_preds)
print("RandomForestRegressor RMSE:", ran_rmse)

RandomForestRegressor RMSE: 18488.19209271583


Because of overfitting issue we need to use cross-validation technique

In [19]:
from sklearn.model_selection import cross_val_score
# Decision Tree with cross-validation
tree_rmses = -cross_val_score(
 dec_reg,
 housing_prep,
 housing_labels,
 scoring="neg_root_mean_squared_error",
 cv=10
)
print("Decision Tree CV RMSEs:", tree_rmses)
print("\nCross-Validation Performance (Decision Tree):")
print(pd.Series(tree_rmses).describe())

Decision Tree CV RMSEs: [70721.34718641 70205.58789008 64442.32744089 69301.03002646
 68349.35602452 69233.85257856 72113.88489376 70052.35488194
 67142.55453673 69547.22533523]

Cross-Validation Performance (Decision Tree):
count       10.000000
mean     69110.952079
std       2113.125296
min      64442.327441
25%      68570.480163
50%      69424.127681
75%      70167.279638
max      72113.884894
dtype: float64


In [22]:
# Random Forest with cross validation
ran_rmses = -cross_val_score(
    ran_reg,
    housing_prep,
    housing_labels,
    scoring = "neg_root_mean_squared_error",
    cv=10
)
print("Random Forest CV RMSEs:", ran_rmses)
print("\nCross-Validation Performance (Random Forest):")
print(pd.Series(ran_rmses).describe()) 

Random Forest CV RMSEs: [50798.56985683 49334.05833764 46132.72713865 50666.72963231
 47010.11845019 49542.37201445 51759.55764759 48878.69787398
 47647.04124653 52968.0783552 ]

Cross-Validation Performance (Random Forest):
count       10.000000
mean     49473.795055
std       2148.919214
min      46132.727139
25%      47954.955403
50%      49438.215176
75%      50765.609801
max      52968.078355
dtype: float64


In [23]:
# randomforest rsme mean    49473.795055     

In [25]:
housing_prep

array([[-0.94135046,  1.34743822,  0.02756357, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.17178212, -1.19243966, -1.72201763, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.26758118, -0.1259716 ,  1.22045984, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-1.5707942 ,  1.31001828,  1.53856552, ...,  0.        ,
         0.        ,  0.        ],
       [-1.56080303,  1.2492109 , -1.1653327 , ...,  0.        ,
         0.        ,  0.        ],
       [-1.28105026,  2.02567448, -0.13148926, ...,  0.        ,
         0.        ,  0.        ]])